=============================================================================
# PROYECTO: DETECCIÓN DE FRAUDE FINANCIERO CON MACHINE LEARNING
# Asignatura: Modelo de Predicción
=============================================================================

# Project Overview

La detección de fraude representa un problema altamente desafiante debido al **desbalanceo extremo de clases**, la alta dimensionalidad y la presencia de grandes cantidades de datos faltantes.

En este proyecto se desarrolla un pipeline completo de **Data Science y Machine Learning**, incluyendo:

- Data Understanding
- Exploratory Data Analysis (EDA)
- Data Preprocessing
- Feature Engineering
- Selección de Variables
- Modelado Predictivo
- Evaluación de Modelos


# Dataset

Se utilizó el dataset **IEEE-CIS Fraud Detection**, compuesto por información transaccional y variables de identidad asociadas a cada operación.

El dataset incluye:

- información de pago,
- datos del dispositivo,
- comportamiento transaccional,
- atributos temporales,
- relaciones entre entidades,
- e indicadores de riesgo anonimizados.


# Data Understanding

## Objetivo

El objetivo de esta fase fue comprender la estructura del dataset, evaluar la calidad de los datos e identificar desafíos relevantes antes del modelado predictivo.

Durante esta etapa se analizaron:

- dimensiones del dataset,
- tipos de variables,
- distribución de fraude,
- consistencia entre datasets,
- valores faltantes,
- variables dominadas,
- y características generales de los datos.


In [4]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(".")

files = [
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
    "sample_submission.csv"
]

for file in files:
    print("\n" + "="*80)
    print(f"ARCHIVO: {file}")
    print("="*80)

    path = DATA_PATH / file

    df = pd.read_csv(path, nrows=1000)

    print("\nDimensiones de la muestra:")
    print(df.shape)

    print("\nPrimeras 5 filas:")
    print(df.head())

    print("\nColumnas:")
    print(df.columns.tolist())

    print("\nTipos de datos:")
    print(df.dtypes)

    print("\nValores faltantes en la muestra:")
    print(df.isna().sum().sort_values(ascending=False).head(30))

    print("\nResumen numérico:")
    print(df.describe().T.head(30))

    print("\nColumnas con pocos valores únicos:")
    for col in df.columns:
        nunique = df[col].nunique(dropna=True)
        if nunique <= 20:
            print(f"{col}: {nunique} valores únicos -> {df[col].dropna().unique()[:20]}")


ARCHIVO: train_transaction.csv

Dimensiones de la muestra:
(1000, 394)

Primeras 5 filas:
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
3  567.0  150.0  mastercard  117.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
4  514.0  150.0  mastercard  102.0  ...  0.0   0.0   0.0   0.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path(".")

files = [
    "train_transaction.csv",
    "train_identity.csv",
    "test_transaction.csv",
    "test_identity.csv",
    "sample_submission.csv"
]

for file in files:
    path = DATA_PATH / file
    
    rows = sum(1 for _ in open(path, encoding="utf-8")) - 1
    cols = len(pd.read_csv(path, nrows=0).columns)
    
    print(f"{file}: {rows:,} filas | {cols:,} columnas")

train_transaction.csv: 590,540 filas | 394 columnas
train_identity.csv: 144,233 filas | 41 columnas
test_transaction.csv: 506,691 filas | 393 columnas
test_identity.csv: 141,907 filas | 41 columnas
sample_submission.csv: 506,691 filas | 2 columnas


In [ ]:
import pandas as pd

train_transaction = pd.read_csv("train_transaction.csv")

print(train_transaction["isFraud"].value_counts())

print("\nPorcentaje:")
print(
    train_transaction["isFraud"]
    .value_counts(normalize=True) * 100
)

isFraud
0    569877
1     20663
Name: count, dtype: int64

Porcentaje:
isFraud
0    96.500999
1     3.499001
Name: proportion, dtype: float64


In [ ]:
train_identity = pd.read_csv("train_identity.csv", nrows=5)
test_identity = pd.read_csv("test_identity.csv", nrows=5)

print("TRAIN:")
print(train_identity.columns.tolist())

print("\nTEST:")
print(test_identity.columns.tolist())

print("\n¿Son iguales?")
print(train_identity.columns.tolist() == test_identity.columns.tolist())

TRAIN:
['TransactionID', 'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']

TEST:
['TransactionID', 'id-01', 'id-02', 'id-03', 'id-04', 'id-05', 'id-06', 'id-07', 'id-08', 'id-09', 'id-10', 'id-11', 'id-12', 'id-13', 'id-14', 'id-15', 'id-16', 'id-17', 'id-18', 'id-19', 'id-20', 'id-21', 'id-22', 'id-23', 'id-24', 'id-25', 'id-26', 'id-27', 'id-28', 'id-29', 'id-30', 'id-31', 'id-32', 'id-33', 'id-34', 'id-35', 'id-36', 'id-37', 'id-38', 'DeviceType', 'DeviceInfo']

¿Son iguales?
False


In [ ]:
import pandas as pd

# =====================================================
# 1. CARGA
# =====================================================

train_transaction = pd.read_csv("train_transaction.csv")
train_identity = pd.read_csv("train_identity.csv")

test_transaction = pd.read_csv("test_transaction.csv")
test_identity = pd.read_csv("test_identity.csv")

# =====================================================
# 2. ARREGLAR COLUMNAS TEST IDENTITY
# =====================================================

test_identity.columns = [
    col.replace("-", "_") if col.startswith("id-") else col
    for col in test_identity.columns
]

# =====================================================
# 3. MERGE
# =====================================================

train = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

test = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
)

print("=" * 80)
print("SHAPES")
print("=" * 80)

print("Train:", train.shape)
print("Test:", test.shape)

# =====================================================
# 4. MISSING VALUES
# =====================================================

missing = pd.DataFrame({
    "missing_count": train.isnull().sum(),
    "missing_pct": train.isnull().mean() * 100
})

missing = missing.sort_values(
    by="missing_pct",
    ascending=False
)

print("\n" + "=" * 80)
print("TOP 50 VARIABLES CON MÁS MISSING")
print("=" * 80)

print(missing.head(50))

# =====================================================
# 5. VARIABLES CONSTANTES
# =====================================================

constant_cols = []

for col in train.columns:
    if train[col].nunique(dropna=False) <= 1:
        constant_cols.append(col)

print("\n" + "=" * 80)
print("VARIABLES CONSTANTES")
print("=" * 80)

print(constant_cols)
print(f"\nTotal: {len(constant_cols)}")

# =====================================================
# 6. VARIABLES CON BAJA VARIABILIDAD
# =====================================================

low_variance = []

for col in train.columns:
    top_freq = train[col].value_counts(
        normalize=True,
        dropna=False
    )

    if len(top_freq) > 0:
        dominance = top_freq.iloc[0]

        if dominance > 0.95:
            low_variance.append(
                (col, round(dominance * 100, 2))
            )

print("\n" + "=" * 80)
print("VARIABLES DOMINADAS (>95% mismo valor)")
print("=" * 80)

print(low_variance[:50])

# =====================================================
# 7. TIPOS DE VARIABLES
# =====================================================

print("\n" + "=" * 80)
print("TIPOS DE DATOS")
print("=" * 80)

print(train.dtypes.value_counts())

SHAPES
Train: (590540, 434)
Test: (506691, 433)

TOP 50 VARIABLES CON MÁS MISSING
       missing_count  missing_pct
id_24         585793    99.196159
id_25         585408    99.130965
id_07         585385    99.127070
id_08         585385    99.127070
id_21         585381    99.126393
id_26         585377    99.125715
id_22         585371    99.124699
id_27         585371    99.124699
id_23         585371    99.124699
dist2         552913    93.628374
D7            551623    93.409930
id_18         545427    92.360721
D13           528588    89.509263
D14           528353    89.469469
D12           525823    89.041047
id_04         524216    88.768923
id_03         524216    88.768923
D6            517353    87.606767
id_33         517251    87.589494
D8            515614    87.312290
D9            515614    87.312290
id_09         515614    87.312290
id_10         515614    87.312290
id_30         512975    86.865411
id_32         512954    86.861855
id_34         512735    86.824771
